<h1>Training a DistilBERT text classifier for the UK Key Stage of literature</h1>

<h2> Step 1. Import the necessary libraries </h2>

In [ ]:
import inspect
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments


def resolve_data_dir() -> Path:
    env_dir = os.getenv('UK_KEY_STAGE_DATA_DIR')
    candidates = []
    if env_dir:
        candidates.append(Path(env_dir).expanduser())

    candidates.extend([
        Path.cwd() / 'database' / 'model1',
        Path.cwd().parent / 'database' / 'model1',
        Path('database/model1'),
        Path('../database/model1'),
    ])

    for candidate in candidates:
        if (candidate / 'TRAIN_balanced.csv').exists() and (candidate / 'TEST_balanced.csv').exists():
            return candidate.resolve()

    raise FileNotFoundError(
        'No se encontraron TRAIN_balanced.csv y TEST_balanced.csv. '
        'Colocalos en database/model1 o define UK_KEY_STAGE_DATA_DIR.'
    )


DATA_DIR = resolve_data_dir()
print(f'Using dataset from: {DATA_DIR}')
print('Available CSV files:', sorted(path.name for path in DATA_DIR.glob('*.csv')))

<h2>Step 2. Load the text and classes from the dataset</h2>

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'TRAIN_balanced.csv')
test_df = pd.read_csv(DATA_DIR / 'TEST_balanced.csv')

train_df = train_df[['segments', 'UK Key Stage']]
test_df = test_df[['segments', 'UK Key Stage']]

print('Training data:')
print(train_df.head(5))

print()
print('Testing data:')
print(test_df.head(5))

<h2>Step 3. Encode the labels for classification</h2>

In [ ]:
label_encoder = LabelEncoder()
train_df['UK Key Stage'] = label_encoder.fit_transform(train_df['UK Key Stage'])
test_df['UK Key Stage'] = label_encoder.transform(test_df['UK Key Stage'])

<h2> Step 4. Set up the Tokeniser, and convert the data for classification </h2>

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


def tokenize_function(examples):
    return tokenizer(examples["segments"], padding="max_length", truncation=True, max_length=512)


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("UK Key Stage", "labels")
test_dataset = test_dataset.rename_column("UK Key Stage", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

num_labels = len(label_encoder.classes_)
print("Number of labels: " + str(num_labels))

<h2>Step 5. Load the DistilBERT model from HuggingFace and define training arguments</h2>

Note that we are training this model with a high batch size for a small number of epochs during this tutorial

In [ ]:
import mlflow
mlflow.set_experiment("UK_Key_Stage_Classification")

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)

training_kwargs = {
    "output_dir": "./results",
    "save_strategy": "epoch",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 2,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy",
    "report_to": "mlflow",
}

training_arg_names = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" in training_arg_names:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": test_dataset,
    "compute_metrics": compute_metrics,
}

trainer_arg_names = set(inspect.signature(Trainer.__init__).parameters)
if "processing_class" in trainer_arg_names:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

<H2>Step 6. Train the model!</H2>

In [ ]:
trainer.train()

<h2>Step 7. Evaluate the model on the test dataset and save the model</h2>

In [ ]:
test_results = trainer.evaluate(test_dataset)
print('Test Results:', test_results)